# PROXORA: Variational Quantum Learning, First Step

This notebook is the exploratory, visual counterpart to `src/proxora/__init__.py`. It keeps the same experiment: optimize one parameter of a one-qubit circuit so that the probability of measuring `|1⟩` approaches a target probability of 0.8.

Unlike the command-line entry point, the notebook separates the experiment into small, inspectable steps and explains the mathematics in Markdown before implementing it. The plots use ordinary Cartesian axes with `θ` on the x-axis and probability or loss on the y-axis.

## 1. The quantum model

We begin in the computational basis state

$$
|0\rangle = \begin{bmatrix}1\\0\end{bmatrix}.
$$

A single-qubit rotation about the **Y axis** is applied. Its matrix is

$$
R_y(\theta) =
\begin{bmatrix}
\cos(\theta/2) & -\sin(\theta/2)\\
\sin(\theta/2) & \cos(\theta/2)
\end{bmatrix}.
$$

Applying the gate to $|0\rangle$ gives the state

$$
|\psi(\theta)\rangle = R_y(\theta)|0\rangle
= \cos(\theta/2)|0\rangle + \sin(\theta/2)|1\rangle.
$$

The Born rule says that a measurement outcome's probability is the squared magnitude of its amplitude. Therefore,

$$
P(1\mid\theta) = |\langle 1|\psi(\theta)\rangle|^2
= \left|\sin(\theta/2)\right|^2
= \sin^2(\theta/2).
$$

This is the ideal probability predicted by the statevector, before finite-shot measurement sampling.

In [ ]:
from math import pi

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize_scalar
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

TARGET_PROBABILITY = 0.8
SHOTS = 1000
INITIAL_THETA = 0.7

plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True})

## 2. Calculate the ideal probability

The function below mirrors the function in `__init__.py`: build a circuit, apply $R_y(\theta)$, obtain its statevector, and read the amplitude associated with $|1\rangle$.

For a one-qubit statevector, index `0` corresponds to $|0\rangle$ and index `1` corresponds to $|1\rangle$. The probability is the squared magnitude of the latter amplitude.

In [ ]:
def probability_of_one(theta: float) -> float:
    """Ideal probability of measuring |1> after an Ry(theta) gate."""
    circuit = QuantumCircuit(1)
    circuit.ry(theta, 0)
    state = Statevector.from_instruction(circuit)
    return float(abs(state.data[1]) ** 2)

initial_probability = probability_of_one(INITIAL_THETA)
print(f"Initial theta: {INITIAL_THETA:.6f} rad")
print(f"Initial ideal P(1): {initial_probability:.6f}")

## 3. Visualize the probability curve

The angle $\theta$ is the horizontal coordinate, in radians. The vertical coordinate is the probability $P(1\mid\theta)$, which must lie between 0 and 1.

Our target is $P(1)=0.8$. The curve shows which rotation angles can produce that probability. We mark the initial parameter to see where optimization begins.

In [ ]:
theta_values = np.linspace(0.0, pi, 500)
probability_values = np.array([probability_of_one(theta) for theta in theta_values])

fig, ax = plt.subplots()
ax.plot(theta_values, probability_values, label=r"$P(1|\theta)=\sin^2(\theta/2)$")
ax.axhline(TARGET_PROBABILITY, linestyle="--", label=f"Target = {TARGET_PROBABILITY:.1f}")
ax.scatter([INITIAL_THETA], [initial_probability], zorder=3, label="Initial parameter")
ax.set(title="Measurement probability versus rotation angle", xlabel=r"Rotation angle $\theta$ (radians)", ylabel=r"Probability $P(1|\theta)$", xlim=(0, pi), ylim=(0, 1))
ax.set_xticks([0, pi / 4, pi / 2, 3 * pi / 4, pi], ["0", r"$\pi/4$", r"$\pi/2$", r"$3\pi/4$", r"$\pi$"])
ax.legend()
plt.show()

## 4. Define the learning objective

We want the model's ideal probability to match the target $p^*=0.8$. Define the error

$$
e(\theta)=P(1\mid\theta)-p^*.
$$

The loss is the squared error:

$$
\boxed{L(\theta)=\left(P(1\mid\theta)-p^*\right)^2
=\left(\sin^2(\theta/2)-0.8\right)^2.}
$$

Squaring makes the loss non-negative and penalizes errors in either direction. A perfect ideal match has $L(\theta)=0$.

In [ ]:
def loss(theta: float) -> float:
    error = probability_of_one(theta) - TARGET_PROBABILITY
    return error**2

loss_values = np.array([loss(theta) for theta in theta_values])
initial_loss = loss(INITIAL_THETA)

fig, ax = plt.subplots()
ax.plot(theta_values, loss_values, label=r"$L(\theta)=(P(1|\theta)-0.8)^2")
ax.scatter([INITIAL_THETA], [initial_loss], zorder=3, label="Initial loss")
ax.set(title="Squared-error loss versus rotation angle", xlabel=r"Rotation angle $\theta$ (radians)", ylabel=r"Loss $L(\theta)$", xlim=(0, pi), ylim=(0, max(loss_values) * 1.05))
ax.set_xticks([0, pi / 4, pi / 2, 3 * pi / 4, pi], ["0", r"$\pi/4$", r"$\pi/2$", r"$3\pi/4$", r"$\pi$"])
ax.legend()
plt.show()

print(f"Initial loss: {initial_loss:.8f}")

## 5. Optimize the parameter

This first experiment uses a **classical optimizer** to choose the circuit parameter. `minimize_scalar` searches the bounded interval $0\leq\theta\leq\pi$.

$$
\theta^* = \underset{0\leq\theta\leq\pi}{\operatorname{argmin}}\;L(\theta).
$$

Because the model is simple, we can also derive the exact ideal solution. Starting from

$$
\sin^2(\theta/2)=p^*,
$$

and taking the solution in $[0,\pi]$ gives

$$
\boxed{\theta^*=2\arcsin\!\left(\sqrt{p^*}\right).}
$$

We use the optimizer as the experimental result and compare it with this analytical value as a check.

In [ ]:
optimization = minimize_scalar(
    loss,
    bounds=(0.0, pi),
    method="bounded",
    options={"xatol": 1e-8},
)

trained_theta = float(optimization.x)
trained_probability = probability_of_one(trained_theta)
analytical_theta = 2 * np.arcsin(np.sqrt(TARGET_PROBABILITY))

print(f"Optimizer success: {optimization.success}")
print(f"Initial theta: {INITIAL_THETA:.6f}")
print(f"Trained theta: {trained_theta:.6f}")
print(f"Analytical theta: {analytical_theta:.6f}")
print(f"Absolute theta difference: {abs(trained_theta - analytical_theta):.3e}")
print(f"Trained ideal P(1): {trained_probability:.6f}")
print(f"Final loss: {optimization.fun:.8e}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(theta_values, loss_values, label="Loss landscape")
ax.scatter([INITIAL_THETA], [loss(INITIAL_THETA)], s=65, zorder=3, label="Initial parameter")
ax.scatter([trained_theta], [loss(trained_theta)], s=65, zorder=3, label="Optimized parameter")
ax.axvline(trained_theta, linestyle="--", alpha=0.8)
ax.set(title="Optimization result on the loss landscape", xlabel=r"Rotation angle $\theta$ (radians)", ylabel=r"Loss $L(\theta)$", xlim=(0, pi), ylim=(0, max(loss_values) * 1.05))
ax.set_xticks([0, pi / 4, pi / 2, 3 * pi / 4, pi], ["0", r"$\pi/4$", r"$\pi/2$", r"$3\pi/4$", r"$\pi$"])
ax.legend()
plt.show()

## 6. From ideal probability to finite-shot measurements

A real measurement run does not return a probability directly. Each shot produces either `0` or `1`. With $N$ independent shots and ideal probability $p=P(1\mid\theta)$, the number of ones is modeled as

$$
K\sim\operatorname{Binomial}(N,p),
\qquad \widehat p=\frac{K}{N}.
$$

Here $N=1000$. The measured estimate $\widehat p$ will usually differ slightly from the ideal probability because of sampling variability. A useful reference for its standard deviation is

$$
\operatorname{SE}(\widehat p)=\sqrt{\frac{p(1-p)}{N}}.
$$

The simulator uses a fixed seed for reproducibility. This is a shot-based simulation, not a claim that the experiment ran on quantum hardware.

In [ ]:
measured_circuit = QuantumCircuit(1, 1)
measured_circuit.ry(trained_theta, 0)
measured_circuit.measure(0, 0)

simulator = AerSimulator()
counts = simulator.run(
    measured_circuit,
    shots=SHOTS,
    seed_simulator=42,
).result().get_counts()

measured_probability = counts.get("1", 0) / SHOTS
standard_error = np.sqrt(trained_probability * (1 - trained_probability) / SHOTS)

print(f"Measured counts: {counts}")
print(f"Shots: {SHOTS}")
print(f"Measured P(1): {measured_probability:.3f}")
print(f"Ideal P(1): {trained_probability:.6f}")
print(f"Approximate standard error: {standard_error:.4f}")
print("Measured circuit:")
print(measured_circuit.draw(output="text"))

## 7. Compare the target, ideal result, and sampled result

The target and ideal probability describe the mathematical model. The measured probability is an empirical estimate from a finite number of shots. Keeping these quantities separate helps avoid treating sampling noise as an optimizer failure.

In [ ]:
fig, ax = plt.subplots()
labels = ["Target", "Ideal after training", "Measured estimate"]
values = [TARGET_PROBABILITY, trained_probability, measured_probability]
x_positions = np.arange(len(labels))
ax.plot(x_positions, values, marker="o", linestyle="none", markersize=8)
ax.axhline(TARGET_PROBABILITY, linestyle="--", label="Target probability")
ax.set(title="Probability comparison", xlabel="Reference", ylabel="Probability", xticks=x_positions, xticklabels=labels, ylim=(0, 1))
ax.legend()
plt.show()

## 8. Interpretation and next experiments

- The circuit has one trainable parameter, $\theta$.
- The probability curve comes directly from the statevector and the Born rule.
- The optimizer minimizes a classical squared-error objective.
- The final shot estimate is stochastic even when the ideal state is known.
- This toy example demonstrates variational optimization mechanics; by itself it does **not** demonstrate quantum advantage or a trained next-value predictor.

A natural next step for PROXORA is to connect the parameterized-circuit experiment to actual sequence data, then evaluate it against a classical baseline on the same held-out samples.